# Python for (Open) Neuroscience  
## Lecture 0.5 — Classes and Objects

Sara Assecondi

Department of Psychology and Cognitive Sciences

   
>**Goal of this lecture:** understand what classes and objects are, why Python uses them everywhere, and how to define simple classes that bundle together data and behavior.

---

## Lecture outline

1. Why objects are useful  
2. Objects: attributes and methods  
3. Classes vs objects  
4. The `__init__()` method and `self`  
5. Inspecting objects  
6. Writing your own methods  
7. Properties  
8. Class attributes vs instance attributes  
9. Optional advanced topics: special methods and inheritance


## Why do we need objects?

Imagine you are working with data from an experiment.

You may have:

- imaging data
- stimulus information
- sampling frequency
- participant metadata
- a set of functions that operate on all of these

One option is to keep everything in separate variables and pass them around repeatedly.


In [3]:
# One entity -> many variables, many functions

imaging_data = [0.1, 0.4, 0.3]
stimulus_data = ["on", "off", "on"]
sampling_frequency = 1000

def resample(imaging_data, sampling_frequency, new_frequency):
    print(f"Resampling from {sampling_frequency} Hz to {new_frequency} Hz")

def crop_on_stimulus(imaging_data, stimulus_data, sampling_frequency, padding=0):
    print(f"Cropping with padding={padding}")

resample(imaging_data, sampling_frequency, new_frequency=250)
crop_on_stimulus(imaging_data, stimulus_data, sampling_frequency, padding=10)


Resampling from 1000 Hz to 250 Hz
Cropping with padding=10


This works, but it is easy to end up with many related variables floating around.

A second option is to keep them in a dictionary.


In [4]:
# One entity -> one dictionary, many functions

experiment = {
    "imaging_data": [0.1, 0.4, 0.3],
    "stimulus_data": ["on", "off", "on"],
    "sampling_frequency": 1000,
}

def resample(experiment, new_frequency):
    print(
        f"Resampling from {experiment['sampling_frequency']} Hz "
        f"to {new_frequency} Hz"
    )

def crop_on_stimulus(experiment, padding=0):
    print(f"Cropping with padding={padding}")

resample(experiment, new_frequency=250)
crop_on_stimulus(experiment, padding=10)


Resampling from 1000 Hz to 250 Hz
Cropping with padding=10


<div style="background:#e6f2ff; padding:20px; border-radius:12px;">This is where <strong>objects</strong> become useful.</div>

## Object-oriented programming (OOP)

**Object-oriented programming** is a programming paradigm based on **objects**: entities that bind together

- **data**  
- **operations on those data**

Objects keep together:

- **Attributes**: values stored inside the object
- **Methods**: functions attached to the object, usually operating on its attributes

You can think of an object as a **dictionary plus behavior**.

**example**: participant data

Below we define a class representing a participant in an experiment.

First focus on the big picture:

- the class is called `ParticipantData`
- each object created from it will store participant-specific information
- the class also defines useful methods

In [7]:
class ParticipantData:
    """Represent data from an experimental participant."""

    def __init__(self, participant_id, age, condition, base_path="/path/to/data"):
        self.participant_id = participant_id
        self.age = age
        self.condition = condition
        self.base_path = base_path
        self.experiment_name = "Experiment 1"

    def get_data_path(self):
        """Generate the file path for this participant's data."""
        return f"{self.base_path}/subject_{self.participant_id}_{self.condition}.csv"

    def get_exp_metadata(self):
        """Return metadata as a dictionary."""
        return {
            "participant_id": self.participant_id,
            "age": self.age,
            "condition": self.condition,
            "experiment_name": self.experiment_name,
        }

    def get_exp_data(self):
        """Pretend to load data from disk."""
        data_path = self.get_data_path()
        print(f"Loading data from {data_path} ...")
        return [1, 2, 3]

    def __repr__(self):
        return (
            f"ParticipantData(participant_id={self.participant_id!r}, "
            f"age={self.age!r}, condition={self.condition!r})"
        )


### Class vs object

This distinction is fundamental:

- A **class** is a blueprint or template
- An **object** is a concrete instance created from that class

A class is like the concept of a chair.  
An object is one specific chair in the room.

In [8]:
# Instantiation: creating objects from the class

participant_a = ParticipantData(participant_id="A", age=28, condition="treatment")
participant_b = ParticipantData(participant_id="B", age=35, condition="control")

participant_a, participant_b


(ParticipantData(participant_id='A', age=28, condition='treatment'),
 ParticipantData(participant_id='B', age=35, condition='control'))

### Attributes

- Attributes are values stored inside the object.
- We access them with dot notation:

```python
object_name.attribute_name
```


In [9]:
print(participant_a.age)
print(participant_a.experiment_name)
print(participant_b.condition)


28
Experiment 1
control


This is conceptually similar to dictionary access, just with different syntax.


In [ ]:
participant_dict = {"participant_id": "A", "age": 28}
print(participant_dict["age"])

print(participant_a.age)


One advantage of classes is that they let us define a **consistent structure**.

Every `ParticipantData` object is expected to have:

- `participant_id`
- `age`
- `condition`
- `base_path`
- `experiment_name`


### Methods

- Methods are functions attached to an object.
- We call them with dot notation as well:

```python
object_name.method_name(...)
```


In [10]:
participant_a.get_data_path()


'/path/to/data/subject_A_treatment.csv'

In [11]:
metadata = participant_a.get_exp_metadata()
metadata

{'participant_id': 'A',
 'age': 28,
 'condition': 'treatment',
 'experiment_name': 'Experiment 1'}

In [12]:
data = participant_a.get_exp_data()
data

Loading data from /path/to/data/subject_A_treatment.csv ...


[1, 2, 3]

<div style="background:#e6f2ff; padding:20px; border-radius:12px;">Notice that many methods do not require us to pass much information. Why? Because the object already contains the relevant attributes.</div>


#### `__init__()`: initialization method

- When you create an object, Python automatically calls the `__init__()` method.
- Its role is to initialize the object: this is where we usually define its attributes.

When you write:

```python
participant_a = ParticipantData("A", 28, "treatment")
```
Python creates a new object and then runs `__init__()` on that object.

#### `self`: the current object

- Inside a class definition, `self` refers to the *specific object* instance being used / created.
- This is why you see `self` as the first argument of methods.

For example:

>`self.age` mean the `age` attribute of *this specific object*

In [14]:
class IntrospectiveClass:
    def __init__(self, attribute=0):
        self.my_attribute = attribute

    def __str__(self):
        return f"IntrospectiveClass(my_attribute={self.my_attribute})"

    def introspect(self):
        print(f"Who am I? {self}")
        print(f"My attribute value is: {self.my_attribute}")

obj1 = IntrospectiveClass(1)
obj2 = IntrospectiveClass(2)

obj1.introspect()
print("---")
obj2.introspect()


Who am I? IntrospectiveClass(my_attribute=1)
My attribute value is: 1
---
Who am I? IntrospectiveClass(my_attribute=2)
My attribute value is: 2


Each object has its own `self`, and therefore its own attribute values.

#### Arguments in `__init__()`

The rules are exactly the same as for functions:

- arguments can be positional or keyword-based
- arguments can have default values
- required arguments must be provided (those without default, as "enrollment")


In [ ]:
# create other Student objects
student_positional = Student("Alex", 24, "Cognitive Science")
student_keyword = Student(name="Alex", age=24, enrollment="Cognitive Science")
student_default = Student("Sam", 25)

In [ ]:
# inspect the objects
print(student_positional.introduce())
print(student_keyword.introduce())
print(student_default.introduce())

#### Methods can also modify the object

- So far our methods mostly returned values.
- Methods can also **change** the internal state of the object.
- **Using .copy()** is a good habit when you do not want the original list outside the object to be modified indirectly.


In [ ]:
# define a class that averages RTs and appends new RTs
class ReactionTimesAnalyzer:
    """Class to analyze reaction time data."""

    def __init__(self, reaction_times):
        self.reaction_times = reaction_times.copy()

    def get_average_rt(self):
        return sum(self.reaction_times) / len(self.reaction_times)

    def append_new_trial(self, reaction_time):
        self.reaction_times.append(reaction_time)

In [22]:
# create a ReactionTimesAnalyzer object
rt = ReactionTimesAnalyzer([0.4, 0.8])
print("Before:", rt.reaction_times)

Before: [0.4, 0.8]


In [23]:
# use a method
rt.append_new_trial(0.6)

print("After: ", rt.reaction_times)
print("Mean:  ", rt.get_average_rt())

After:  [0.4, 0.8, 0.6]
Mean:   0.6


Because we used `.copy()` in the class, the input is not modified

In [24]:
original_list = [0.4, 0.8]
rt = ReactionTimesAnalyzer(original_list)

rt.append_new_trial(1.2)

print("Object data:  ", rt.reaction_times)
print("Original list:", original_list)


Object data:   [0.4, 0.8, 1.2]
Original list: [0.4, 0.8]


#### Special methods 
*dunder methods (double underscore)*: special methods that allow Python to interact with objects through its built-in syntax and functions.

|Method|Description|
|---|---|
|`__init__`| how the object is set up |
|`__repr__` / `__str__` | how the object is shown (printable result)|
|`__len__` | how big it is|
|`__getitem__` | how to access parts of it|
|`__eq__` | how to compare it|
|`__iter__` | how to loop over it|

#### Objects in Python

- In Python, **everything is an object**.
- You have already been using methods on objects since the beginning.
- Classes and objects are **foundation** of how Python works.


In [15]:
text = "Some text"
print(type(text))
print(text.split())

<class 'str'>
['Some', 'text']


In [16]:
a_list = [1, 2, 3]
a_list.append(4)
a_list

[1, 2, 3, 4]

In [17]:
a_number = 5
print(type(a_number))
print(a_number.bit_length())

<class 'int'>
3


## Inspecting objects

Sometimes it is useful to inspect what an object contains.

Common tools:

- `vars(obj)` or `obj.__dict__` → instance attributes
- `dir(obj)` → attributes and methods
- `getattr(obj, "name")` → access by name
- `callable(...)` → check whether something can be called


In [ ]:
vars(participant_a)


In [ ]:
participant_a.__dict__


In [32]:
dir(participant_a)[:20]


['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__']

In [33]:
getattr(participant_a, "age")


28

In [34]:
for attr_name in ["age", "get_exp_data"]:
    attr = getattr(participant_a, attr_name)
    print(f"{attr_name!r}: callable? -> {callable(attr)}")


'age': callable? -> False
'get_exp_data': callable? -> True


### A simple class

Let us build a simple class and focus on structure.

In [ ]:
# define a simple class Student (the template)
class Student:
    """A very small class representing a student."""

    def __init__(self, name, age, enrollment="CIMeC PhD"):
        self.name = name
        self.age = age
        self.enrollment = enrollment

    def introduce(self):
        return f"My name is {self.name}, I am {self.age}, and I am enrolled in {self.enrollment}."

In [20]:
# create two Student objects
student_1 = Student("Pippo", 26)
student_2 = Student(name="Peppa", age=27, enrollment="Neuroscience MSc")

In [19]:
# inspect the objects
print(student_1.introduce())
print(student_2.introduce())

My name is Pippo, I am 26, and I am enrolled in CIMeC PhD.
My name is Peppa, I am 27, and I am enrolled in Neuroscience MSc.


### Properties

A **property** is a method that behaves like an attribute.

This is useful when:

- a value should be computed on demand
- you want attribute-like syntax
- you want to protect or validate access


In [ ]:
class BehavioralExperiment:
    def __init__(self, reaction_times):
        self.reaction_times = reaction_times.copy()

    def append_new_trial_rt(self, reaction_time):
        self.reaction_times.append(reaction_time)

    @property
    def mean_rt(self):
        if len(self.reaction_times) == 0:
            return None
        return sum(self.reaction_times) / len(self.reaction_times)

exp = BehavioralExperiment([0.45, 0.52, 0.38, 0.41])
print(f"Mean RT: {exp.mean_rt:.3f} s")

exp.append_new_trial_rt(2.1)
print(f"Updated mean RT: {exp.mean_rt:.3f} s")


<div style="background:#e6f2ff; padding:20px; border-radius:12px;">
Notice the syntax:

```python
exp.mean_rt
```   
not

```python
exp.mean_rt()
```

That is because `mean_rt` is a property, not a regular method.
</div>

A **getter** property is used **to read** a value.   
A **setter** property is used **to change** a value.

So:

>**getter** = what happens when you access **obj.x**

>**setter** = what happens when you assign **obj.x = something**

### Read-only properties and protected attributes

By convention, a leading underscore means:

> this attribute is intended for internal use

It is not truly private, but it signals that users should not modify it directly.


In [ ]:
class SubjectRecord:
    def __init__(self, subject_id, reaction_times):
        self._subject_id = subject_id
        self.reaction_times = reaction_times.copy()

    @property
    def subject_id(self):
        return self._subject_id

subject = SubjectRecord("S01", [1.0, 1.2, 0.9])
print(subject.subject_id)


Because `subject_id` is exposed only through a getter property, assigning to it directly will fail.


In [ ]:
try:
    subject.subject_id = "S99"
except AttributeError as e:
    print("Error:", e)


You *can* still access the underscored attribute directly:


In [ ]:
subject._subject_id


But by convention, you should avoid doing that from outside the class unless you truly know what you are doing.


### Properties for validation

Properties can also define validation logic when setting a value.


In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self._age = age

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("Age cannot be negative.")
        self._age = value

In [ ]:
person = Person("Alex", 20)
print(person.age)

person.age = 21
print(person.age)

In [ ]:
try:
    person.age = -1
except ValueError as e:
    print("Error:", e)

### Instance attributes vs class attributes

So far we have mostly used **instance attributes**:
```python
self.age
self.name
```

These belong to each object separately.   
So **different objects** can have **different values**.

A **class attribute** is defined at the class level.   
So **all objects** of that class can access the **same shared value**.


In [29]:
class DemoClass:
    class_attribute = []

    def __init__(self):
        self.instance_attribute = []

# create DemoClass objects
obj1 = DemoClass()
obj2 = DemoClass()

In [30]:
# print instant attribute
obj2.instance_attribute.append(1)
print("Instance attributes:")
print(obj1.instance_attribute, obj2.instance_attribute)

Instance attributes:
[] [1]


In [31]:
# print instant attribute
obj2.class_attribute.append(99)
print("\nClass attributes:")
print(obj1.class_attribute, obj2.class_attribute)


Class attributes:
[99] [99]


>This is a classic source of bugs.

>Be especially careful with **mutable class attributes** such as lists and dictionaries.


## Inheritance

**Inheritance** lets us define a more specific class starting from a more general one.

For example:

- `Person` → general concept
- `Student` → more specific kind of person

`super()` lets us call methods from the parent class instead of rewriting them from scratch.


In [ ]:
# define class
class PersonBase:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        print(f"My name is {self.name} and I am {self.age} years old.")

In [ ]:
# define "derived" class
class StudentPerson(PersonBase):
    def __init__(self, name, age, student_id):
        super().__init__(name, age)
        self.student_id = student_id

    def introduce(self):
        super().introduce()
        print(f"My student ID is {self.student_id}.")

In [ ]:
# inspect objects
student = StudentPerson("Bob", 15, "128654")
student.introduce()

## Optional advanced topic: a custom special method

Python lets classes emulate built-in behavior.

For example, `__getitem__()` makes an object indexable with square brackets.


In [ ]:
class MATLABList:
    def __init__(self, items):
        self.items = items

    def __getitem__(self, item_idx):
        if item_idx <= 0:
            raise IndexError("Indexing in MATLABList starts from 1!")
        return self.items[item_idx - 1]

shop_list = MATLABList(["bread", "salt", "coffee"])
print(shop_list[1])
print(shop_list[3])


This is mostly for illustration: just because you *can* redefine behavior does not always mean you *should*.


## *args and **kwargs

`*args` and `**kwargs` are useful when we want a function or method to accept a flexible number of inputs.

They are especially common in classes when a child class wants to pass arguments to a parent class without explicitly listing every one of them.

**The main idea**
- `*args` collects extra **positional arguments** into a tuple
- `**kwargs` collects extra **keyword arguments** into a dictionary

So they act like *“catch-all”* containers for additional inputs.

`*args`: extra positional arguments

In [35]:
def arbitrary_inputs_function(first_argument, *args):
    print(f"Captured argument: {first_argument}")
    print(type(args), args)

arbitrary_inputs_function(1, 2, 3)

Captured argument: 1
<class 'tuple'> (2, 3)


Here:

- first_argument gets 1
- *args collects the remaining positional arguments, 2 and 3, into a tuple

`**kwargs`: extra keyword arguments

In [ ]:
def arbitrary_inputs_function(first_argument, *args, **kwargs):
    print(f"Captured argument: {first_argument}")
    print(f"args ({type(args)}): {args}")
    print(f"kwargs ({type(kwargs)}): {kwargs}")

arbitrary_inputs_function(1, 2, 3, random_kwarg=5)

Here:

- `*args` stores extra positional arguments
- `**kwargs` stores extra keyword arguments

## A compact checklist for reading class-based code

When you encounter a new class, ask:

1. What is the class trying to represent?  
2. What attributes are set in `__init__()`?  
3. Which methods are available?  
4. Which methods return values, and which modify the object?  
5. Are there properties?  
6. Is the class inheriting from another class?  
7. Are there class attributes shared across instances?


## Final note

At the start, classes may feel abstract. That is normal.

The key is to keep the central idea in mind:

> **an object bundles together state and behavior**

Once that clicks, many parts of Python become easier to read.
